In [1]:
%pip install langchain_ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter

d:\nihal\masters\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
doc="""Artificial Intelligence (AI) is transforming many industries by enabling computers to perform tasks that traditionally required human intelligence. 
In healthcare, AI assists doctors by analyzing medical images, predicting diseases, and recommending treatment plans. 
In education, AI-powered tutors provide personalized learning experiences based on a student's progress. 
Businesses use AI to automate customer support through chatbots, optimize supply chains, and detect fraudulent transactions. 
However, AI also raises ethical concerns, including data privacy, algorithmic bias, and job displacement due to automation.
To ensure responsible AI adoption, organizations should prioritize transparency, fairness, accountability, and human oversight while developing and deploying AI systems."""

question="What are the ethical concerns associated with AI, and what principles should organizations follow to ensure responsible AI adoption?"

In [4]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=30)
split=text_splitter.split_text(doc)

In [5]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_texts(texts=split, 
                                    embedding=embeddings)

retriever = vectorstore.as_retriever()

In [6]:
import os, getpass
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith key: ")

from langsmith import Client
client = Client()
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

In [7]:
from langchain_core.runnables import RunnablePassthrough
llm = ChatOllama(model="llama3.1:8b", temperature=0)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question — use the AI-ethics question that matches the document
rag_chain.invoke(question)

'The ethical concerns associated with AI include data privacy, algorithmic bias, and job displacement due to automation. To address these concerns, organizations should prioritize transparency, fairness, accountability, and human oversight when developing and deploying AI systems. By following these principles, organizations can ensure responsible AI adoption and mitigate potential negative consequences.'